In [176]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

rng = np.random.default_rng(100)

def purchase_probability(price):
    price = np.asarray(price, dtype=float)
    return (
        0.55 * np.exp(-np.logaddexp(0.0, (price - 13.9) / 0.8))
        + 0.30 * np.exp(-np.logaddexp(0.0, (price - 30.9) / 1.2))
        + 0.15 * np.exp(-np.logaddexp(0.0, (price - 58.4) / 1.8))
    )

def sales(price=14.0, rng=None, mean_customers=30):
    price = float(price)
    if not np.isfinite(price):
        raise ValueError("price must be finite")
    if price < 0:
        raise ValueError("price must be nonnegative")
    mean_customers = float(mean_customers)
    if not np.isfinite(mean_customers) or mean_customers < 0:
        raise ValueError("mean_customers must be finite and nonnegative")
    rng = np.random.default_rng(rng)
    customer_count = rng.poisson(mean_customers)
    probability = purchase_probability(price)
    return rng.binomial(1, probability, size=customer_count).astype(np.int8)

# Initial test of difference with 1000 individuals to predict n for future attempts

## get about 500 people each and bootstrap 

In [177]:


p_A = 10.00
p_B = 15.00
size = 500
power_n=[]

sales_A = sales(p_A, mean_customers=size, rng=rng) * p_A # Compute revenue for each customer in A
sales_B = sales(p_B, mean_customers=size, rng=rng) * p_B # Compute revenue for each customer in B
print(f'There are {len(sales_A)} customers in A and {len(sales_B)} customers in B')

k=2000
for i in range(k):
    boot_sales_a=rng.choice(sales_A, size=len(sales_A), replace=True)
    boot_sales_b=rng.choice(sales_B, size=len(sales_B), replace=True)

    revenue_A = boot_sales_a.mean() # Mean revenue per arriving customer in A
    revenue_B = boot_sales_b.mean() # Mean revenue per arriving customer in B

    delta_R = revenue_B - revenue_A # Difference in mean revenue per arriving customer

    #Lehr's power model
    power_estimation=16*(np.var(boot_sales_a, ddof=1)+np.var(boot_sales_b, ddof=1))/(delta_R**2)
    #round up
    power_n.append(int(power_estimation)+1)

print(f"Using the Lehr's model for sample size, to have an 80% power rate, our n should be about {np.mean(power_n)}")

There are 525 customers in A and 474 customers in B
Using the Lehr's model for sample size, to have an 80% power rate, our n should be about 467.015


# Next, we do a grid search of 10-70 dollars in $5 increments with n=500

## draw them, then bootstrap to simulate the experiment many times

In [178]:
grid=np.linspace(start=10, stop=70, num=13)

In [179]:
draw=[]
size = 500
for i in range(len(grid)):
    draw.append([sales(grid[i], mean_customers=size, rng=rng) * grid[i]])
draw_df=pd.DataFrame(draw, columns=['draws'])
draw_df['price']=grid

draw_df

,draws,price
0,"[10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10....",10.0
1,"[15.0, 0.0, 15.0, 15.0, 0.0, 0.0, 0.0, 0.0, 0....",15.0
2,"[0.0, 0.0, 20.0, 0.0, 20.0, 0.0, 0.0, 20.0, 20...",20.0
3,"[0.0, 0.0, 0.0, 0.0, 0.0, 25.0, 0.0, 25.0, 0.0...",25.0
4,"[0.0, 0.0, 0.0, 0.0, 30.0, 30.0, 0.0, 0.0, 0.0...",30.0
5,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",35.0
6,"[0.0, 0.0, 40.0, 0.0, 0.0, 40.0, 0.0, 0.0, 0.0...",40.0
7,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",45.0
8,"[0.0, 0.0, 50.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",50.0
9,"[0.0, 0.0, 55.0, 0.0, 0.0, 0.0, 0.0, 0.0, 55.0...",55.0


In [180]:

bestprice=10
secondbestprice=0

for i in range(len(grid)):
    p_A = bestprice
    p_B = grid[i]
    sales_A = list(draw_df.loc[draw_df['price']==bestprice]['draws'])[0]
    sales_B = list(draw_df.loc[draw_df['price']==grid[i]]['draws'])[0] 
    k=5000
    bestprice_boot=[]

    for j in range(k):
        boot_sales_a=rng.choice(sales_A, size=len(sales_A), replace=True)
        boot_sales_b=rng.choice(sales_B, size=len(sales_B), replace=True)
    
        revenue_A = boot_sales_a.mean() # Mean revenue per arriving customer in A
        revenue_B = boot_sales_b.mean() # Mean revenue per arriving customer in B
        delta_R = revenue_B - revenue_A # Difference in mean revenue per arriving customer
        std_error = np.sqrt(
            np.var(sales_A, ddof=1) / len(sales_A) +
            np.var(sales_B, ddof=1) / len(sales_B)
        ) # Standard Error

        confidence_interval = (delta_R - 1.96 * std_error, delta_R + 1.96 * std_error)

        #determine which price is better
        if confidence_interval[0]<=0 and confidence_interval[1]<=0:
            #print(f'{p_A} is better than {p_B}')
            bestprice_boot.append(p_A)
            continue
        elif confidence_interval[0]>=0 and confidence_interval[1]>=0 == np.True_:
            bestprice_boot.append(p_B)
            #print(f'{p_B} is better than {p_A}')
            
        else:
            #print(f'Could not decide between {p_A} and {p_B}')
            #bestprice_boot.append(p_B)
            continue
    bestprice = np.median(bestprice_boot)
    if not bestprice>0:
        print(f"could not determine difference between {p_A} and {p_B}. Moving on.")
        bestprice=p_B
        continue
    else:
        print(f'Between {p_A} and {p_B}, the better price is {bestprice}')
        continue
            

Between 10 and 10.0, the better price is 10.0
Between 10.0 and 15.0, the better price is 10.0
Between 10.0 and 20.0, the better price is 10.0


c:\Users\5maxc\anaconda3\envs\ds6001_remade\Lib\site-packages\numpy\_core\fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\5maxc\anaconda3\envs\ds6001_remade\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


could not determine difference between 10.0 and 25.0. Moving on.
Between 25.0 and 30.0, the better price is 25.0
Between 25.0 and 35.0, the better price is 25.0
Between 25.0 and 40.0, the better price is 25.0
Between 25.0 and 45.0, the better price is 25.0
Between 25.0 and 50.0, the better price is 25.0
Between 25.0 and 55.0, the better price is 25.0
Between 25.0 and 60.0, the better price is 25.0
Between 25.0 and 65.0, the better price is 25.0
Between 25.0 and 70.0, the better price is 25.0


# Spot check between 10 and 25 to decide which to investigate further.

## Not only will we compare 10 and 25, we will also do a power analysis to see how big a sample we would need for 80% power of a test

In [181]:
p_A = 10
p_B = 25
size=2000
sales_A = sales_A = sales(p_A, mean_customers=size, rng=rng) * p_A # Compute revenue for each customer in A
sales_B = sales(p_B, mean_customers=size, rng=rng) * p_B # Compute revenue for each customer in B
k=5000
bestprice_boot=[]
power_n=[]

for j in range(k):
    boot_sales_a=rng.choice(sales_A, size=len(sales_A), replace=True)
    boot_sales_b=rng.choice(sales_B, size=len(sales_B), replace=True)

    revenue_A = boot_sales_a.mean() # Mean revenue per arriving customer in A
    revenue_B = boot_sales_b.mean() # Mean revenue per arriving customer in B
    delta_R = revenue_B - revenue_A # Difference in mean revenue per arriving customer
    std_error = np.sqrt(
        np.var(sales_A, ddof=1) / len(sales_A) +
        np.var(sales_B, ddof=1) / len(sales_B)
    ) # Standard Error

    confidence_interval = (delta_R - 1.96 * std_error, delta_R + 1.96 * std_error)
    #Lehr's power model
    power_estimation=16*(np.var(boot_sales_a, ddof=1)+np.var(boot_sales_b, ddof=1))/(delta_R**2)
    #round up
    power_n.append(int(power_estimation)+1)

    #determine which price is better
    if confidence_interval[0]<=0 and confidence_interval[1]<=0:
        #print(f'{p_A} is better than {p_B}')
        bestprice_boot.append(p_A)
        continue
    elif confidence_interval[0]>=0 and confidence_interval[1]>=0 == np.True_:
        bestprice_boot.append(p_B)
        #print(f'{p_B} is better than {p_A}')
        
    else:
        #print(f'Could not decide between {p_A} and {p_B}')
        #bestprice_boot.append(p_B)
        continue
bestprice = np.median(bestprice_boot)
print(f'the best price between 10 and 25 is {bestprice}')
print(f"Using the Lehr's model for sample size, to have an 80% power rate, our n should be about {np.mean(power_n)}")

the best price between 10 and 25 is nan
Using the Lehr's model for sample size, to have an 80% power rate, our n should be about 173243.8064


# Since we still could not figure it out even with n=2000 we will try a finer grid surrounding both 10 and 25 and then compare the best from each.

## Start with 5-15

In [182]:
grid=np.linspace(start=5, stop=15, num=11)
draw=[]
size = 500
for i in range(len(grid)):
    draw.append([sales(grid[i], mean_customers=size, rng=rng) * grid[i]])
draw_df=pd.DataFrame(draw, columns=['draws'])
draw_df['price']=grid

In [183]:
bestprice=10


for i in range(len(grid)):
    p_A = bestprice
    p_B = grid[i]
    sales_A = list(draw_df.loc[draw_df['price']==bestprice]['draws'])[0]
    sales_B = list(draw_df.loc[draw_df['price']==grid[i]]['draws'])[0] 
    k=5000
    bestprice_boot=[]

    for j in range(k):
        boot_sales_a=rng.choice(sales_A, size=len(sales_A), replace=True)
        boot_sales_b=rng.choice(sales_B, size=len(sales_B), replace=True)
    
        revenue_A = boot_sales_a.mean() # Mean revenue per arriving customer in A
        revenue_B = boot_sales_b.mean() # Mean revenue per arriving customer in B
        delta_R = revenue_B - revenue_A # Difference in mean revenue per arriving customer
        std_error = np.sqrt(
            np.var(sales_A, ddof=1) / len(sales_A) +
            np.var(sales_B, ddof=1) / len(sales_B)
        ) # Standard Error

        confidence_interval = (delta_R - 1.96 * std_error, delta_R + 1.96 * std_error)

        #determine which price is better
        if confidence_interval[0]<=0 and confidence_interval[1]<=0:
            #print(f'{p_A} is better than {p_B}')
            bestprice_boot.append(p_A)
            continue
        elif confidence_interval[0]>=0 and confidence_interval[1]>=0 == np.True_:
            bestprice_boot.append(p_B)
            #print(f'{p_B} is better than {p_A}')
            
        else:
            #print(f'Could not decide between {p_A} and {p_B}')
            #bestprice_boot.append(p_B)
            continue
    bestprice = np.median(bestprice_boot)
    if not bestprice>0:
        print(f"could not determine difference between {p_A} and {p_B}. Moving on.")
        print(f'Top revenue is {sales_A.mean()}')
        bestprice=p_B
        continue
    else:
        print(f'Between {p_A} and {p_B}, the better price is {bestprice}')
        continue

print('\n')
print(f'Top revenue is {sales_A.mean()}')

Between 10 and 5.0, the better price is 10.0
Between 10.0 and 6.0, the better price is 10.0
Between 10.0 and 7.0, the better price is 10.0
Between 10.0 and 8.0, the better price is 10.0
Between 10.0 and 9.0, the better price is 10.0
Between 10.0 and 10.0, the better price is 10.0
could not determine difference between 10.0 and 11.0. Moving on.
Top revenue is 9.92619926199262
could not determine difference between 11.0 and 12.0. Moving on.
Top revenue is 10.8698224852071
Between 12.0 and 13.0, the better price is 12.0
Between 12.0 and 14.0, the better price is 12.0
Between 12.0 and 15.0, the better price is 12.0


Top revenue is 11.473282442748092


## Next 20-30

In [184]:
grid=np.linspace(start=20, stop=30, num=11)
draw=[]
size = 500
for i in range(len(grid)):
    draw.append([sales(grid[i], mean_customers=size, rng=rng) * grid[i]])
draw_df=pd.DataFrame(draw, columns=['draws'])
draw_df['price']=grid

In [185]:
bestprice=20
average_rev_for_all=[]

for i in range(len(grid)):
    p_A = bestprice
    p_B = grid[i]
    sales_A = list(draw_df.loc[draw_df['price']==bestprice]['draws'])[0]
    sales_B = list(draw_df.loc[draw_df['price']==grid[i]]['draws'])[0] 
    k=5000
    bestprice_boot=[]
    average_rev_a=[]
    average_rev_b=[]

    for j in range(k):
        boot_sales_a=rng.choice(sales_A, size=len(sales_A), replace=True)
        boot_sales_b=rng.choice(sales_B, size=len(sales_B), replace=True)
    
        revenue_A = boot_sales_a.mean() # Mean revenue per arriving customer in A
        revenue_B = boot_sales_b.mean() # Mean revenue per arriving customer in B
        average_rev_a.append(revenue_A)  #append current price average revenue
        average_rev_b.append(revenue_B)  #append current price average revenue
        delta_R = revenue_B - revenue_A # Difference in mean revenue per arriving customer

        std_error = np.sqrt(
            np.var(sales_A, ddof=1) / len(sales_A) +
            np.var(sales_B, ddof=1) / len(sales_B)
        ) # Standard Error

        confidence_interval = (delta_R - 1.96 * std_error, delta_R + 1.96 * std_error)

        #determine which price is better
        if confidence_interval[0]<=0 and confidence_interval[1]<=0:
            #print(f'{p_A} is better than {p_B}')
            bestprice_boot.append(p_A)
            continue
        elif confidence_interval[0]>=0 and confidence_interval[1]>=0 == np.True_:
            bestprice_boot.append(p_B)
            #print(f'{p_B} is better than {p_A}')
            
        else:
            #print(f'Could not decide between {p_A} and {p_B}')
            #bestprice_boot.append(p_B)
            continue
    print(f'The average revenue for {p_B} is {np.mean(average_rev_b)}')
    average_rev_for_all.append(np.mean(average_rev_b))
    bestprice = np.median(bestprice_boot)
    if not bestprice>0:
        print(f"could not determine difference between {p_A} and {p_B}. Moving on.")
        print(f'Top revenue is price {p_A} with {np.mean(average_rev_a)}')
        bestprice=p_B
        continue
    else:
        print(f'Between {p_A} and {p_B}, the better price is {bestprice}')
        continue

print('\n')
print(f'Top revenue is {np.max(average_rev_for_all)}')


The average revenue for 20.0 is 9.656656488549618
Between 20 and 20.0, the better price is 20.0
The average revenue for 21.0 is 9.823236363636365
Between 20.0 and 21.0, the better price is 20.0
The average revenue for 22.0 is 9.888762162162163
Between 20.0 and 22.0, the better price is 20.0
The average revenue for 23.0 is 10.023475255623723
Between 20.0 and 23.0, the better price is 20.0
The average revenue for 24.0 is 11.286647657841142
could not determine difference between 20.0 and 24.0. Moving on.
Top revenue is price 20.0 with 9.656419847328243
The average revenue for 25.0 is 11.590147928994082
Between 24.0 and 25.0, the better price is 24.0
The average revenue for 26.0 is 11.143679238095238
Between 24.0 and 26.0, the better price is 24.0
The average revenue for 27.0 is 11.839000415800415
Between 24.0 and 27.0, the better price is 24.0
The average revenue for 28.0 is 12.453517575757575
Between 24.0 and 28.0, the better price is 24.0
The average revenue for 29.0 is 11.1073625
Betwe

# Finally, a tight grid around 26 and 27

## To do that, we need to have a power estimation again.

In [186]:
p_A = 26.00
p_B = 27.00
size = 500
power_n=[]

sales_A = sales(p_A, mean_customers=size, rng=rng) * p_A # Compute revenue for each customer in A
sales_B = sales(p_B, mean_customers=size, rng=rng) * p_B # Compute revenue for each customer in B
print(f'There are {len(sales_A)} customers in A and {len(sales_B)} customers in B')

k=2000
for i in range(k):
    boot_sales_a=rng.choice(sales_A, size=len(sales_A), replace=True)
    boot_sales_b=rng.choice(sales_B, size=len(sales_B), replace=True)

    revenue_A = boot_sales_a.mean() # Mean revenue per arriving customer in A
    revenue_B = boot_sales_b.mean() # Mean revenue per arriving customer in B

    delta_R = revenue_B - revenue_A # Difference in mean revenue per arriving customer

    #Lehr's power model
    power_estimation=16*(np.var(boot_sales_a, ddof=1)+np.var(boot_sales_b, ddof=1))/(delta_R**2)
    #round up
    power_n.append(int(power_estimation)+1)

print(f"Using the Lehr's model for sample size, to have an 80% power rate, our n should be about {np.mean(power_n)}")

There are 536 customers in A and 524 customers in B
Using the Lehr's model for sample size, to have an 80% power rate, our n should be about 3511952314.8525


## Uh oh, that's a bit too expensive...

Since to have any meaningful statistical difference, we estimate that we would need an absurd amount of data which is not feasible. Thus, we conclude that the highest revenue is most likely somewhere between pricing at 25-28 with expected revenue of $ \$ 11.5$ through $ \$12.5$ per person
